# K-Means Clustering (Digits + Iris) – Solution

**Short name (GitHub):** `KMDigits`

Worked answers for `KMDigits_Practice_Skeleton.ipynb`. Numbers below were measured with `sklearn.cluster.KMeans(n_clusters=10, n_init=10, random_state=0)` on `load_digits()` (or the bundled CSV).

| Card | inertia | purity (majority map) | ARI |
|------|---------|------------------------|-----|
| Iris k=3 | 78.85 | 0.893 | 0.730 |
| Digits k=10 | 1,165,189 | 0.792 | 0.666 |

Cluster ids are **not** digit labels. After fitting you must map each centroid to its majority true digit before you talk about “accuracy.”


## Inline cheat-sheet (keep this cell visible)

See also **`KMDigits_Cheatsheet.docx`**.

| Item | Formula / code |
|------|----------------|
| Sample vs feature | row = one image / flower; column = one pixel / measurement |
| Distance | \(d(x,c)=\sqrt{\sum_j(x_j-c_j)^2}\) — assign to nearest centroid |
| Assignment | `labels = ((X[:,None,:]-C[None,:,:])**2).sum(2).argmin(1)` |
| Update | `C[j] = X[labels==j].mean(axis=0)` |
| Inertia | \(J=\sum_i\|x_i-c_{\ell_i}\|^2\) — always falls as \(k\) grows |
| Elbow | plot \(J(k)\); pick the bend, not the minimum |
| sklearn | `KMeans(n_clusters=k).fit(X)` then `.predict` / `.labels_` / `.cluster_centers_` / `.inertia_` |
| Convergence | centroids move \(<\texttt{tol}\) or `max_iter` |
| Purity | after majority-vote map of cluster → true class |
| ARI | chance-adjusted pair agreement; 1 = perfect, 0 ≈ random |
| Inference | new 64-vector → nearest of the 10 learned centroids |

**Order that matters:** `fit` before `predict`. Shortcut: `fit_predict`.


## Desired outcome

![flowchart](kmeans_digits_flowchart.png)

1. Load unlabeled samples × features.
2. Choose \(k\) (domain knowledge, or elbow).
3. Place \(k\) centroids (random or k-means++).
4. Assign every row to its nearest centroid.
5. Replace each centroid with the mean of its rows.
6. Repeat 4–5 until the shift is below `tol`.
7. Inspect centroid images, map clusters → digits, score, simulate knobs.


## 0. Packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import image as mpimg

try:
    from sklearn.datasets import load_digits, load_iris
    from sklearn.cluster import KMeans, MiniBatchKMeans
    from sklearn.decomposition import PCA
    from sklearn.metrics import adjusted_rand_score, confusion_matrix
    HAS_SK = True
except ImportError:
    HAS_SK = False
    from KMDigits import KMeans, adjusted_rand as adjusted_rand_score
    print("sklearn missing — using KMDigits.KMeans")

from KMDigits import (
    KMeans as KMScratch,
    assign,
    inertia,
    map_clusters_to_truth,
    elbow_curve,
)

rng = np.random.default_rng(0)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = False
print("sklearn available:", HAS_SK)


## 1. Unsupervised clustering in one paragraph

Labeled answers are missing. K-means answers two questions: how many groups (`k`), and what does “similar” mean (Euclidean distance to a centroid). Training = assign + update until the centroids stop walking. Inference = nearest centroid.

## 2. Load Iris (warm-up) and Digits (the project)

Iris: 150 samples, 4 features (cm), 3 species. Digits: 1,797 samples, 64 pixels, 10 digits. Targets exist so we can *check* a clustering. They do not go into `.fit`.

In [ ]:
# Prefer sklearn loaders; fall back to the bundled CSVs.
try:
    iris_bunch = load_iris()
    X_iris = iris_bunch.data
    y_iris = iris_bunch.target
    iris_names = list(iris_bunch.feature_names)
except Exception:
    iris_df = pd.read_csv("data/iris.csv")
    X_iris = iris_df.drop(columns=["species"]).to_numpy()
    y_iris = iris_df["species"].to_numpy()
    iris_names = list(iris_df.columns[:-1])

try:
    digits = load_digits()
    X = digits.data            # (1797, 64)
    y = digits.target          # 0..9
    images = digits.images     # (1797, 8, 8)
except Exception:
    dig_df = pd.read_csv("data/digits.csv")
    X = dig_df.drop(columns=["digit"]).to_numpy()
    y = dig_df["digit"].to_numpy()
    images = X.reshape(-1, 8, 8)

print("iris", X_iris.shape, "digits", X.shape, "pixel range", X.min(), X.max())
print("digit counts", np.bincount(y))
print("DESCR snippet: 8x8 images, pixels 0-16, 1797 samples from 30 writers (UCI / Alpaydin).")


## 3. Look at the data before you cluster

In [ ]:
# Two-feature view so we can see the three species before clustering.
fig, ax = plt.subplots()
sc = ax.scatter(X_iris[:, 2], X_iris[:, 3], c=y_iris, cmap="viridis", s=28, edgecolor="k", linewidth=0.3)
ax.set_xlabel("petal length (cm)")
ax.set_ylabel("petal width (cm)")
ax.set_title("Iris — labeled species (ground truth, not used by k-means)")
plt.colorbar(sc, ax=ax, ticks=[0, 1, 2], label="species")
plt.tight_layout()
plt.savefig("kmeans_iris_truth.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(8, 3.4))
for ax, idx in zip(axes.ravel(), range(10)):
    ax.imshow(images[idx], cmap="gray_r")
    ax.set_title(f"idx {idx} → {y[idx]}", fontsize=9)
    ax.axis("off")
fig.suptitle("First 10 digit images (8×8, pixels 0–16)")
plt.tight_layout()
plt.savefig("kmeans_digits_samples.png", dpi=120, bbox_inches="tight")
plt.show()
print("index 100 is a", y[100], "— Codecademy asks you to confirm it looks like a 4.")


## 4. From-scratch k-means (the hard way)

Four functions: place centroids, assign, update, loop. This is the Codecademy “implement it yourself” block, vectorized.

In [ ]:
def random_centroids(X, k, seed=0):
    r = np.random.default_rng(seed)
    return X[r.choice(len(X), size=k, replace=False)].copy()

def assign_labels(X, centroids):
    # (n, 1, d) - (1, k, d) → (n, k)
    d2 = ((X[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
    return d2.argmin(axis=1)

def update_centroids(X, labels, k):
    C = np.zeros((k, X.shape[1]))
    for j in range(k):
        mask = labels == j
        C[j] = X[mask].mean(axis=0) if mask.any() else X[np.random.randint(len(X))]
    return C

def kmeans_loop(X, k, max_iter=30, seed=0, tol=1e-4):
    C = random_centroids(X, k, seed)
    history = []
    for t in range(1, max_iter + 1):
        labels = assign_labels(X, C)
        new_C = update_centroids(X, labels, k)
        shift = np.linalg.norm(new_C - C)
        history.append((t, float(((X - new_C[labels]) ** 2).sum()), shift))
        if shift < tol:
            C = new_C
            break
        C = new_C
    labels = assign_labels(X, C)
    return C, labels, history

C3, lab3, hist3 = kmeans_loop(X_iris, k=3, seed=0)
print("scratch iris inertia", round(hist3[-1][1], 2), "iters", hist3[-1][0])
print("first 3 history rows", hist3[:3])


## 5. Iris with sklearn — confirm k=3

In [ ]:
model_iris = KMeans(n_clusters=3, n_init=10, random_state=0)
model_iris.fit(X_iris)
print("sklearn iris inertia", round(model_iris.inertia_, 2), "n_iter", model_iris.n_iter_)
mapped_i, map_i, pur_i = map_clusters_to_truth(model_iris.labels_, y_iris, 3)
ari_i = adjusted_rand_score(y_iris, model_iris.labels_)
print("purity", round(pur_i, 4), "ARI", round(ari_i, 4), "cluster→species", map_i)

fig, ax = plt.subplots()
ax.scatter(X_iris[:, 2], X_iris[:, 3], c=model_iris.labels_, cmap="viridis", s=28, edgecolor="k", linewidth=0.3)
ax.scatter(model_iris.cluster_centers_[:, 2], model_iris.cluster_centers_[:, 3],
           c="red", s=120, marker="X", label="centroids")
ax.set_xlabel("petal length"); ax.set_ylabel("petal width")
ax.set_title("Iris k=3 — cluster ids (not species ids)")
ax.legend()
plt.tight_layout()
plt.savefig("kmeans_iris_clusters.png", dpi=120, bbox_inches="tight")
plt.show()


## 6. Digits with sklearn — k=10

In [ ]:
k = 10  # ten digits
model = KMeans(n_clusters=k, n_init=10, random_state=0)
model.fit(X)
print("digits inertia", round(model.inertia_, 1), "n_iter", model.n_iter_)
print("cluster sizes", np.bincount(model.labels_))


## 7. Centroids as pictures

A centroid lives in the same 64-D space as a sample, so it *is* an 8×8 image.

In [ ]:
fig = plt.figure(figsize=(8, 3.2))
fig.suptitle("Cluster centers as 8×8 images (k=10)")
for i, c in enumerate(model.cluster_centers_):
    ax = fig.add_subplot(2, 5, i + 1)
    ax.imshow(c.reshape(8, 8), cmap="gray_r")
    ax.set_title(f"cluster {i}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.savefig("kmeans_digits_centroids.png", dpi=120, bbox_inches="tight")
plt.show()
print("Look at the blobs: each center is an 'average handwriting' of its members.")
print("Typical mapping with random_state=0 will NOT be identity — cluster 2 may look like a 7.")


## 8. Map cluster ids to digits + score

In [ ]:
mapped, mapping, purity = map_clusters_to_truth(model.labels_, y, k=10)
ari = adjusted_rand_score(y, model.labels_)
print("cluster → majority digit", mapping)
print("purity", round(purity, 4), "ARI", round(ari, 4))

ct = pd.crosstab(pd.Series(y, name="true digit"),
                 pd.Series(model.labels_, name="cluster"))
print(ct)

fig, ax = plt.subplots(figsize=(6.5, 5))
im = ax.imshow(ct.values, cmap="Blues")
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xlabel("cluster id"); ax.set_ylabel("true digit")
ax.set_title("Digits — true label vs raw cluster id")
for i in range(10):
    for j in range(10):
        ax.text(j, i, ct.values[i, j], ha="center", va="center", fontsize=7)
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig("kmeans_digits_crosstab.png", dpi=120, bbox_inches="tight")
plt.show()


## 9. Elbow method

Inertia always drops with k. The elbow is a *candidate*, not a proof. For digits the domain already says k=10.

In [ ]:
ks = list(range(1, 16))
inertias = []
for kk in ks:
    m = KMeans(n_clusters=kk, n_init=5, random_state=0)
    m.fit(X)
    inertias.append(m.inertia_)

fig, ax = plt.subplots()
ax.plot(ks, inertias, marker="o")
ax.axvline(10, color="crimson", ls="--", label="domain k=10")
ax.set_xlabel("k"); ax.set_ylabel("inertia")
ax.set_title("Elbow — digits (n_init=5)")
ax.legend()
plt.tight_layout()
plt.savefig("kmeans_digits_elbow.png", dpi=120, bbox_inches="tight")
plt.show()
print(list(zip(ks, [round(v, 0) for v in inertias])))
print("Inertia keeps falling after k=10. The digit story, not the elbow alone, justifies k=10.")


## 10. PCA view (64-D → 2-D)

In [ ]:
pca = PCA(n_components=2, random_state=0)
Z = pca.fit_transform(X)
print("PCA explained", pca.explained_variance_ratio_.round(3), "sum", pca.explained_variance_ratio_.sum().round(3))

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
axes[0].scatter(Z[:, 0], Z[:, 1], c=y, cmap="tab10", s=8, alpha=0.7)
axes[0].set_title("PCA — colored by true digit")
axes[1].scatter(Z[:, 0], Z[:, 1], c=model.labels_, cmap="tab10", s=8, alpha=0.7)
axes[1].set_title("PCA — colored by k-means cluster")
for ax in axes:
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
plt.tight_layout()
plt.savefig("kmeans_digits_pca.png", dpi=120, bbox_inches="tight")
plt.show()


## 11. Alternate code that reaches the same idea

In [ ]:
# Alternate A — k-means++ is the sklearn default; contrast with random init.
m_rand = KMeans(n_clusters=10, init="random", n_init=10, random_state=0).fit(X)
m_pp = KMeans(n_clusters=10, init="k-means++", n_init=10, random_state=0).fit(X)
print("random init inertia", round(m_rand.inertia_, 1),
      "k-means++ inertia", round(m_pp.inertia_, 1))

# Alternate B — MiniBatchKMeans (faster on large n)
m_mb = MiniBatchKMeans(n_clusters=10, n_init=10, random_state=0, batch_size=256).fit(X)
print("minibatch inertia", round(m_mb.inertia_, 1),
      "purity", round(map_clusters_to_truth(m_mb.labels_, y, 10)[2], 4))

# Alternate C — the bundled from-scratch class
m_sc = KMScratch(n_clusters=10, n_init=5, random_state=0).fit(X)
print("scratch inertia", round(m_sc.inertia_, 1),
      "purity", round(map_clusters_to_truth(m_sc.labels_, y, 10)[2], 4))

# Alternate D — fit_predict one-liner
labels_fp = KMeans(n_clusters=10, n_init=10, random_state=0).fit_predict(X)
print("fit_predict agrees with .labels_?", np.array_equal(labels_fp, model.labels_))


## 12. More practice

In [ ]:
# Practice P1 — iris on petals only (columns 2:4)
Xp = X_iris[:, 2:4]
mp = KMeans(n_clusters=3, n_init=10, random_state=0).fit(Xp)
print("P1 petals-only purity", round(map_clusters_to_truth(mp.labels_, y_iris, 3)[2], 4),
      "ARI", round(adjusted_rand_score(y_iris, mp.labels_), 4))

# Practice P2 — digits with the wrong k
for kk in (5, 12):
    mk = KMeans(n_clusters=kk, n_init=5, random_state=0).fit(X)
    print(f"P2 k={kk} inertia={mk.inertia_:.0f} purity={map_clusters_to_truth(mk.labels_, y, kk)[2]:.3f}")

# Practice P3 — 2-D pixels: column 20 (a mid stroke) vs column 44
X2 = X[:, [20, 44]]
m2 = KMeans(n_clusters=10, n_init=10, random_state=0).fit(X2)
print("P3 two-pixel purity", round(map_clusters_to_truth(m2.labels_, y, 10)[2], 4),
      "(expect a collapse — two pixels are not a digit)")


## 13. New handwritten samples (`test.html`)

Codecademy’s drawing widget compresses four 80×80 canvases to 4×64 arrays (8×8 blocks averaged). Pixel scale will not match 0–16 exactly — that is part of why a 1990s 30-writer model misses modern handwriting.

In [ ]:
# Four held-out images from the official set, pretending we drew them in test.html.
# Open test.html, draw four digits, click Get Array, and replace new_samples.
new_idx = [100, 200, 300, 400]
new_samples = X[new_idx]
print("true labels of the four stand-ins", y[new_idx])

new_raw = model.predict(new_samples)
new_digits = [mapping[int(c)] for c in new_raw]
print("raw cluster ids", new_raw)
print("mapped digits  ", new_digits)

fig, axes = plt.subplots(1, 4, figsize=(7, 2.2))
for ax, img, tru, pred in zip(axes, new_samples, y[new_idx], new_digits):
    ax.imshow(img.reshape(8, 8), cmap="gray_r")
    ax.set_title(f"true {tru} → pred {pred}", fontsize=9)
    ax.axis("off")
fig.suptitle("Inference on four new 8×8 samples")
plt.tight_layout()
plt.show()


## 14. Simulation — turn the knobs

In [ ]:
# Knobs you can turn. Each setting refits k-means on a (possibly noisy) copy of X.

def run_once(k=10, n=1797, noise=0.0, n_init=10, seed=0):
    r = np.random.default_rng(seed)
    idx = r.choice(len(X), size=min(n, len(X)), replace=False)
    Xs = X[idx].astype(float) + r.normal(0.0, noise, size=(len(idx), X.shape[1]))
    ys = y[idx]
    m = KMeans(n_clusters=k, n_init=n_init, random_state=seed).fit(Xs)
    _, _, pur = map_clusters_to_truth(m.labels_, ys, k)
    return {"k": k, "n": len(idx), "noise": noise, "n_init": n_init,
            "inertia": m.inertia_, "purity": pur,
            "ARI": float(adjusted_rand_score(ys, m.labels_))}

# Baseline + four perturbations
grid = [
    dict(k=10, n=1797, noise=0.0, n_init=10, seed=0),
    dict(k=6,  n=1797, noise=0.0, n_init=10, seed=0),
    dict(k=10, n=400,  noise=0.0, n_init=10, seed=0),
    dict(k=10, n=1797, noise=3.0, n_init=10, seed=0),
    dict(k=10, n=1797, noise=0.0, n_init=1,  seed=0),
]
sim = pd.DataFrame([run_once(**g) for g in grid])
print(sim.round(3))

# Sweep k for a purity/inertia panel
rows = [run_once(k=kk, n=1797, noise=0.0, n_init=5, seed=0) for kk in range(2, 16)]
sweep = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
axes[0].plot(sweep["k"], sweep["inertia"], marker="o")
axes[0].set_title("Simulation — inertia vs k")
axes[0].set_xlabel("k"); axes[0].set_ylabel("inertia")
axes[1].plot(sweep["k"], sweep["purity"], marker="o", color="teal")
axes[1].axhline(0.10, color="gray", ls=":", label="chance (1/10)")
axes[1].set_title("Simulation — purity vs k")
axes[1].set_xlabel("k"); axes[1].legend()
plt.tight_layout()
plt.savefig("kmeans_digits_simulation.png", dpi=120, bbox_inches="tight")
plt.show()


## Audience rewrite (Jočys checklist + McMurrey types)

| Audience | What they need | One sentence they should hear |
|----------|----------------|-------------------------------|
| Expert (ML / vision) | inertia, ARI, init, spherical-cluster caveat | k=10 k-means on 8×8 pixels reaches purity ≈0.79 / ARI ≈0.67; 1 vs 8 and 4 vs 9 share a centroid blob. |
| Technician (OCR / mail-sort) | centroid images, how to paste a new 64-vector | Drop a new glyph on the ten average digits; the nearest average is the sort bin. |
| Executive (ops / product) | cost avoided, error mode | A first-pass clusterer routes most handwritten ZIP strokes without a human keyer; similar-looking digits still collide. |
| Nonspecialist | no jargon | The computer groups pictures that look alike and then names each group after the digit that shows up most. |

Data literacy: experts get the elbow plot; everyone else gets the ten centroid pictures.

Subject knowledge: do not explain “ZIP code” to a postal engineer. Do explain that cluster id ≠ digit until you map it.

**What not to say**
- “The model reads handwriting.” It groups pixels and we name the groups.
- “79% accurate like a classifier.” There was no train/test classification objective.
- “k=10 is proven by the elbow.” The elbow is soft; ten is a domain choice.


## What this model can and cannot do

**Can**
- Segment unlabeled 8×8 glyphs into compact spherical groups.
- Produce a visual prototype (the centroid) for each group.
- Assign a *new* 64-pixel vector to the nearest prototype in milliseconds.
- Warm up a discussion of USPS / check / form OCR — the historical motivation.

**Cannot**
- Respect stroke order, slant, or a 28×28 MNIST canvas without a new matrix.
- Separate interlocking shapes (k-means assumes blobs, not manifolds).
- Give calibrated “this is a 7 with 91% probability.”
- Replace a supervised CNN on modern mail volume.

**Top applications of the same idea**
1. Mail / ZIP-code block grouping
2. Check-amount glyph bins
3. Form-checkbox / handwritten-field pre-sort
4. Customer segmentation (k groups of spenders)
5. Image-color quantization
6. Document topic seeds before LDA
7. Anomaly bins on sensor snapshots
8. Store-layout / planogram clustering
9. Single-cell expression prototypes
10. Thumbnail “similar photos” folders

**Anti-applications**
- Diagnosing disease from one lab row
- Credit decisioning
- Anything where a missed 1-vs-7 is a legal event and you have labels — use a classifier.


## Next steps

- Scale pixels to \([0,1]\) and refit — inertia changes units, purity usually does not.
- Try \(k=9\) after merging the two most confused digits.
- Swap Euclidean for cosine on \(\ell_2\)-normalized rows.
- Read sklearn’s own digits k-means example (`plot_kmeans_digits`).
- Reusable template: `KMDigits_Reusable_Template.ipynb`.
